# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imnotparama/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

Working dir: /content/flyrank-ml-internship
(30000, 44)


**Method choice:** My lane is a ranking problem ("which pages to review
first"), so per the training-honest-models guide, I need a classifier's
probability evaluated at precision@K, not just a label. I'll compare
Logistic Regression (readable baseline model) against Random Forest
(stronger, less readable) — starting simple, adding complexity only if
it earns its keep. Both will be scored with Precision@50, the same
metric my ML-07 baseline rule used, so the comparison is honest.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
from sklearn.model_selection import GroupShuffleSplit

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "search_volume"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(X_train)} rows, {df['client_id'].iloc[train_idx].nunique()} clients")
print(f"Test: {len(X_test)} rows, {df['client_id'].iloc[test_idx].nunique()} clients")

Train: 22885 rows, 24 clients
Test: 7115 rows, 8 clients


**Split design:** Grouped by client_id, not random. This ensures no
client's pages appear in both train and test — otherwise the model
could partly "memorize" a client's patterns instead of generalizing,
which would make the score look better than it honestly is. 25% of
clients held out for testing.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Baseline: same rule as ML-07, scored only on the test set
test_df = df.iloc[test_idx].copy()
test_df["baseline_score"] = (
    (test_df["days_since_last_update"] >= 180).astype(int) *
    (test_df["impressions_90d"] >= 100).astype(int) *
    test_df["impressions_90d"]
)
baseline_p50 = precision_at_k(test_df["baseline_score"], y_test, 50)

# Logistic Regression
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]
logreg_p50 = precision_at_k(logreg_scores, y_test, 50)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(rf_scores, y_test, 50)

base_rate = y_test.mean()

comparison = pd.DataFrame({
    "Method": ["Base rate", "Baseline rule (ML-07)", "Logistic Regression", "Random Forest"],
    "Precision@50": [base_rate, baseline_p50, logreg_p50, rf_p50]
})
print(comparison)

                  Method  Precision@50
0              Base rate      0.516514
1  Baseline rule (ML-07)      0.620000
2    Logistic Regression      0.660000
3          Random Forest      0.580000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# Feature importance (Random Forest)
importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Feature importance:\n", importance)

# 3 concrete wrong cases: model says high-risk, but label says not declining
test_df["rf_score"] = rf_scores
wrong_cases = test_df[(test_df["rf_score"] > 0.7) & (y_test == 0)].head(3)
print("\n3 wrong cases (model confident, but not actually declining):")
print(wrong_cases[["content_id", "rf_score", "days_since_last_update",
                     "impressions_90d", "trend_direction"]])

Feature importance:
 impressions_90d           0.257237
avg_position              0.229009
content_age_days          0.148430
word_count                0.145549
ctr                       0.116680
search_volume             0.065908
days_since_last_update    0.037187
dtype: float64

3 wrong cases (model confident, but not actually declining):
              content_id  rf_score  days_since_last_update  impressions_90d  \
26  content_72c5c2d73e5a      0.83                      13             2426   
34  content_55f75c034970      0.82                       8             3998   
78  content_dea0d86223f3      0.77                      92               59   

   trend_direction  
26          stable  
34              up  
78              up  


**Where the model is most wrong:** Looking at the 3 highest-confidence
wrong cases, all three are pages the Random Forest scored 0.77-0.83
(high decline risk) despite trend_direction being "stable" or "up,"
not "down." Two of these (content_72c5c2d73e5a, content_55f75c034970)
have very low days_since_last_update (13 and 8 days — recently updated)
but high impressions, suggesting the model over-weighs high impressions
as a decline signal even when the page was just refreshed. This mirrors
the exact blind spot found in my ML-07 baseline review: pages can get
flagged based on volume/staleness alone, ignoring whether they're
actually trending down.

**What it leans on:** The top features are impressions_90d (0.257) and
avg_position (0.229), followed by content_age_days (0.148) and
word_count (0.146). These make intuitive sense — high-traffic,
poorly-positioned, older pages are plausible decline candidates. None
looks suspiciously perfect (no single feature dominates above ~90%),
so I don't see obvious leakage here.

**Comparison to baseline:** Surprisingly, Random Forest (0.580)
underperformed both the baseline rule (0.620) and Logistic Regression
(0.660) on this grouped, client-held-out split — the opposite of Week
1's ungrouped result. This is a real, honest finding: with only 8 test
clients and 7 features, Random Forest may be overfitting to training
client patterns that don't transfer, while the simpler Logistic
Regression generalizes better here. This doesn't mean Random Forest is
worse in general — it means grouped validation is a much harder, more
honest test, and complexity isn't automatically rewarded. Logistic
Regression is currently the strongest honest choice on this split.

**Does it fix the ML-07 blind spot?** Partially. The wrong-case review
shows the model still sometimes ignores trend_direction directly
(flagging "stable" and "up" pages), similar to the baseline's
weakness — trend_direction itself isn't a feature (correctly excluded
as leakage-adjacent), so the model has to infer decline risk from
proxies alone, which explains why this gap persists.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.